# KV260 Vitis AI 3.5 Unified Benchmark

Mirrors the structure of the VAI 2.5 benchmark notebook but adapted for
**Vitis AI 3.5 runtime on KV260** (Kria-PYNQ + DPU-PYNQ `design_contest_3.5`).

### Key differences from the 2.5 notebook

| Aspect | 2.5 notebook | 3.5 notebook (this one) |
|---|---|---|
| Stack | DPU-PYNQ 2.5 / VAI 2.5 | DPU-PYNQ design_contest_3.5 / VAI 3.5 |
| DPU IP | DPUCZDX8G_ISA1_B4096 | DPUCZDX8G_ISA1_B4096 (unchanged) |
| DPU fingerprint | 0x101000016010406 | 0x101000056010407 |
| Models | manually pre-staged | host-staged via `scripts/host/04_stage_benchmark.sh` |
| Datasets | manually pre-staged | host-staged via `scripts/host/04_stage_benchmark.sh` |
| Detection accuracy | COCO mAP only | COCO mAP + VOC mAP |
| Pre-flight check | none | **smoke test** filters non-working models before full benchmark |

### About "VAI 3.5 models" on KV260

AMD's VAI 3.5 model zoo doesn't ship KV260 pre-compiled binaries for most
models — the 3.5 binaries target VEK280 and V70. The KV260's DPU IP
(DPUCZDX8G_ISA1_B4096) is **unchanged** between VAI 3.0 and VAI 3.5, so
[per AMD's own documentation](https://wiki.trenz-electronic.de/display/PD/Compilation+of+AI+3.0+models+for+Vitis+2023.2,+AI+3.5+SW,+AI+3.0+DPUCZDX8G),
VAI 3.0 KV260 binaries (`-r3.0.0.tar.gz`) run correctly on the VAI 3.5
runtime. This notebook downloads the VAI 3.0 KV260 binaries and benchmarks
them on the VAI 3.5 runtime.

### Sections

1. **Configuration** — paths, sample sizes (matches 2.5 defaults)
2. **Model catalogue** — every VAI 3.0 model with a KV260 binary
3. **Prerequisite check** — verifies data staged from host (no in-notebook downloads)
4. **System sanity check** — USB autosuspend, etc.
5. **Power monitor**
6. **Preprocessing helpers** (+ VOC additions)
7. **ImageNet / COCO / VOC dataset loaders**
8. **DPU overlay** (loaded once)
9. **Decoders** — YOLO, SSD-TF, RefineDet, EfficientDet, mAP, NMS
10. **Dispatch table** — preprocessing + decoder per model
11. **Smoke test** — single-image gate, excludes non-working models
12. **Main unified benchmark** — power, latency, FPS, accuracy
13. **COCO mAP loop**
14. **VOC mAP loop**
15. **Combined report**
16. **Quick inline view**

**Resume support**: each loop checks its own CSV; already-processed entries
are skipped. Delete a CSV to redo that section.


## 1. Clear DPU state

If a previous notebook left the DPU programmed, unload + clear handles
first. Same as the 2.5 notebook.


In [ ]:
# %run clear.py    # uncomment if you have a clear.py script
# Otherwise just restart the kernel if you see "No Devices Found" later
print("Reminder: if any errors below mention 'No Devices Found',")
print("  - run `sudo xmutil unloadapp` from a board terminal, then re-run the cells")


## 2. Imports and configuration

Defaults match the 2.5 notebook for direct comparison:

| Constant | Value | What it controls |
|---|---|---|
| `N_ACC_IMAGES` | 200 | ImageNet images used for Top-1 accuracy |
| `N_COCO_IMAGES` | 5000 | COCO val2017 images for mAP (full val2017) |
| `N_VOC_IMAGES` | 4952 | VOC2007 test images for mAP (full test set) |
| `CAMERA_DURATION` | 8 | seconds of live camera per model |
| `N_LAT_ITER` | 30 | pure-DPU iterations for latency stats |

To do a quick development run, set `N_COCO_IMAGES = 200` and `N_VOC_IMAGES
= 200`. The catalogue + smoke test still cover every model; only the mAP
loops shrink.


In [ ]:
import os, sys, gc, time, glob, re, csv, json, traceback, subprocess
import hashlib, urllib.request, tarfile, zipfile, shutil
from pathlib import Path
import numpy as np
from PIL import Image

try:
    import cv2
except ImportError as e:
    raise SystemExit("cv2 (opencv-python) is required") from e

ROOT             = Path.cwd()

# Models go in a SEPARATE directory so this doesn't conflict with the
# VAI 2.5 benchmark's Models/ folder.
MODELS_DIR       = ROOT / "Models_VAI35"

# Datasets directory — shared structure but separate from old Dataset/
# in case you have differently-staged data there.
DATASET_DIR      = ROOT / "Datasets"
IMAGENET_IMAGES  = DATASET_DIR / "imagenet_sample" / "images"
IMAGENET_LABELS  = DATASET_DIR / "imagenet_sample" / "labels.txt"
COCO_FULL_DIR    = DATASET_DIR / "coco_val2017"
COCO_IMG_DIR     = COCO_FULL_DIR / "val2017"
COCO_ANN_FILE    = COCO_FULL_DIR / "annotations" / "instances_val2017.json"
VOC_DIR          = DATASET_DIR / "voc2007_test"
VOC_IMG_DIR      = VOC_DIR / "VOCdevkit" / "VOC2007" / "JPEGImages"
VOC_ANN_DIR      = VOC_DIR / "VOCdevkit" / "VOC2007" / "Annotations"
VOC_SPLIT_FILE   = VOC_DIR / "VOCdevkit" / "VOC2007" / "ImageSets" / "Main" / "test.txt"

# Output files
RESULTS_CSV      = ROOT / "vai35_benchmark_results.csv"
COCO_MAP_CSV     = ROOT / "vai35_coco_map_results.csv"
VOC_MAP_CSV      = ROOT / "vai35_voc_map_results.csv"
SMOKE_CSV        = ROOT / "vai35_smoke_test.csv"
REPORT_MD        = ROOT / "vai35_benchmark_report.md"

# Sample counts — matched to the 2.5 notebook so cross-version comparisons
# are like-for-like.
N_WARMUP        = 5
N_LAT_ITER      = 30
N_ACC_IMAGES    = 200
N_COCO_IMAGES   = 5000
N_VOC_IMAGES    = 4952

# mAP / NMS standard thresholds
NMS_IOU_THRESH   = 0.45
CONF_THRESH      = 0.05
MAX_DET_PER_IMG  = 100

# Camera benchmark (BRIO, same as 2.5)
CAMERA_DURATION = 8
CAMERA_DEVICE   = 0
CAMERA_BUFFER   = 4
CAMERA_FOURCC   = 'MJPG'
CAMERA_PRESETS = [
    (640,  480,  60),
    (1280, 720,  30),
    (1920, 1080, 30),
    (3840, 2160, 30),
]

# Expected DPU fingerprint for KV260 VAI 3.5
EXPECTED_DPU_FINGERPRINT = 0x101000056010407

# Make output dirs
for d in (MODELS_DIR, DATASET_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"Models root: {MODELS_DIR}")
print(f"Datasets:    {DATASET_DIR}")
print(f"Outputs:     {RESULTS_CSV.name}, {COCO_MAP_CSV.name}, "
      f"{VOC_MAP_CSV.name}, {SMOKE_CSV.name}, {REPORT_MD.name}")


## 3. Model catalogue

Every VAI 3.0 model with a known KV260 binary that's relevant to a generic
camera benchmark. Format: `<name>: dict(url, category, dataset, decoder,
input_size, mean, scale, channel_order, notes)`.

The URL pattern is:

    https://www.xilinx.com/bin/public/openDownload?filename=<model_name>-zcu102_zcu104_kv260-r3.0.0.tar.gz

> **License note**: some models are marked "Non-Commercial Use Only" by
> AMD ([license details](https://github.com/Xilinx/Vitis-AI/blob/master/model_zoo/AMD-license-agreement-for-non-commercial-models.md)).
> The catalogue includes them but you must comply with the relevant AMD
> license terms when using them. Look for `non_commercial: True` entries.

To add a model: append an entry with the right `decoder` from
`{'classification', 'ssd_tf', 'yolo', 'yolo_voc', 'refinedet', 'efficientdet'}`.
If you don't know the decoder, set it to `None` — the model will still be
benchmarked for latency/power/FPS, just not for accuracy.

To exclude a model: set `enabled: False` in its dict.


In [ ]:
# Helper to keep the catalogue compact
def m(category, dataset=None, decoder='classification',
      input_size=224, mean=None, scale=None, ch='BGR',
      enabled=True, non_commercial=False, notes=''):
    return dict(category=category, dataset=dataset, decoder=decoder,
                input_size=input_size, mean=mean, scale=scale, ch=ch,
                enabled=enabled, non_commercial=non_commercial, notes=notes)

# Standard preprocessing defaults
# ImageNet TF-style: mean [104, 117, 123] (BGR), no scale (already in 0-255)
IM_MEAN_BGR_TF  = [104.0, 117.0, 123.0]
# ImageNet PyTorch: mean [0.485, 0.456, 0.406] * 255 = [123.7, 116.3, 103.5] RGB
IM_MEAN_RGB_PT  = [123.675, 116.28, 103.53]
IM_SCALE_PT     = [0.01712, 0.0175, 0.01743]   # 1/(std*255), std=[0.229, 0.224, 0.225]
# Inception preprocessing: divide by 127.5, subtract 1.0
INC_MEAN        = [127.5, 127.5, 127.5]
INC_SCALE       = [1.0/127.5]*3

CATALOG = {
    # ============================================================
    # CLASSIFICATION (ImageNet, 224×224 standard)
    # ============================================================
    'resnet50':               m('classification', 'imagenet',
                                 mean=IM_MEAN_BGR_TF, ch='BGR'),
    'resnet_v1_50_tf':        m('classification', 'imagenet',
                                 mean=IM_MEAN_BGR_TF, ch='BGR'),
    'resnet_v1_101_tf':       m('classification', 'imagenet',
                                 mean=IM_MEAN_BGR_TF, ch='BGR'),
    'resnet_v1_152_tf':       m('classification', 'imagenet',
                                 mean=IM_MEAN_BGR_TF, ch='BGR'),
    'inception_v1_tf':        m('classification', 'imagenet',
                                 mean=INC_MEAN, scale=INC_SCALE, ch='BGR'),
    'inception_v2_tf':        m('classification', 'imagenet',
                                 mean=INC_MEAN, scale=INC_SCALE, ch='BGR',
                                 input_size=224),
    'inception_v3_tf':        m('classification', 'imagenet',
                                 mean=INC_MEAN, scale=INC_SCALE, ch='BGR',
                                 input_size=299),
    'inception_v4_2016_09_09_tf': m('classification', 'imagenet',
                                 mean=INC_MEAN, scale=INC_SCALE, ch='BGR',
                                 input_size=299),
    'mobilenet_v1_0_25_128_tf': m('classification', 'imagenet',
                                 mean=INC_MEAN, scale=INC_SCALE, ch='BGR',
                                 input_size=128),
    'mobilenet_v1_1_0_224_tf': m('classification', 'imagenet',
                                 mean=INC_MEAN, scale=INC_SCALE, ch='BGR'),
    'mobilenet_v2_1_0_224_tf': m('classification', 'imagenet',
                                 mean=INC_MEAN, scale=INC_SCALE, ch='BGR'),
    'mobilenet_v2_1_4_224_tf': m('classification', 'imagenet',
                                 mean=INC_MEAN, scale=INC_SCALE, ch='BGR'),
    'mobilenetv2_pt':         m('classification', 'imagenet',
                                 mean=IM_MEAN_RGB_PT, scale=IM_SCALE_PT, ch='RGB'),
    'squeezenet_pt':          m('classification', 'imagenet',
                                 mean=IM_MEAN_RGB_PT, scale=IM_SCALE_PT, ch='RGB'),
    'vgg_16_tf':              m('classification', 'imagenet',
                                 mean=IM_MEAN_BGR_TF, ch='BGR'),
    'vgg_19_tf':              m('classification', 'imagenet',
                                 mean=IM_MEAN_BGR_TF, ch='BGR'),
    'efficientnet-b0_tf2':    m('classification', 'imagenet',
                                 mean=INC_MEAN, scale=INC_SCALE, ch='RGB',
                                 input_size=224),
    'efficientnet_edgetpu-S_tf':  m('classification', 'imagenet',
                                 mean=INC_MEAN, scale=INC_SCALE, ch='RGB',
                                 input_size=224),
    'efficientnet_edgetpu-M_tf':  m('classification', 'imagenet',
                                 mean=INC_MEAN, scale=INC_SCALE, ch='RGB',
                                 input_size=240),
    'efficientnet_edgetpu-L_tf':  m('classification', 'imagenet',
                                 mean=INC_MEAN, scale=INC_SCALE, ch='RGB',
                                 input_size=300),
    'inception_resnet_v2_tf': m('classification', 'imagenet',
                                 mean=INC_MEAN, scale=INC_SCALE, ch='BGR',
                                 input_size=299),
    'resnet50_pt':            m('classification', 'imagenet',
                                 mean=IM_MEAN_RGB_PT, scale=IM_SCALE_PT, ch='RGB'),
    'resnet50_pruned_0_4_pt': m('classification', 'imagenet',
                                 mean=IM_MEAN_RGB_PT, scale=IM_SCALE_PT, ch='RGB',
                                 notes='40% pruned'),
    'ofa_resnet50_0_9B_pt':   m('classification', 'imagenet',
                                 mean=IM_MEAN_RGB_PT, scale=IM_SCALE_PT, ch='RGB',
                                 notes='Once-for-All ResNet50'),

    # ============================================================
    # DETECTION — COCO
    # ============================================================
    'ssd_mobilenet_v1_coco_tf':    m('detection', 'coco', decoder='ssd_tf',
                                      input_size=300, mean=INC_MEAN,
                                      scale=INC_SCALE, ch='RGB'),
    'ssd_mobilenet_v2_coco_tf':    m('detection', 'coco', decoder='ssd_tf',
                                      input_size=300, mean=INC_MEAN,
                                      scale=INC_SCALE, ch='RGB'),
    'ssdlite_mobilenetv2_coco_tf': m('detection', 'coco', decoder='ssd_tf',
                                      input_size=300, mean=INC_MEAN,
                                      scale=INC_SCALE, ch='RGB'),
    'ssd_inception_v2_coco_tf':    m('detection', 'coco', decoder='ssd_tf',
                                      input_size=300, mean=INC_MEAN,
                                      scale=INC_SCALE, ch='RGB'),
    'ssd_resnet_50_fpn_coco_tf':   m('detection', 'coco', decoder='ssd_tf',
                                      input_size=640,
                                      mean=[123.68, 116.78, 103.94],
                                      ch='RGB',
                                      notes='ResNet-50 FPN; 640×640 input'),
    'yolov3_coco_416_tf2':         m('detection', 'coco', decoder='yolo',
                                      input_size=416, scale=[1/255]*3, ch='RGB'),
    'yolov4_leaky_spp_m':          m('detection', 'coco', decoder='yolo',
                                      input_size=416, scale=[1/255]*3, ch='RGB'),
    'yolov3':                      m('detection', 'coco', decoder='yolo',
                                      input_size=416, scale=[1/255]*3, ch='RGB',
                                      notes='Darknet YOLOv3 trained on COCO'),
    'efficientdet_d2_tf':          m('detection', 'coco', decoder='efficientdet',
                                      input_size=768, mean=INC_MEAN,
                                      scale=INC_SCALE, ch='RGB',
                                      enabled=False,  # complex decoder; smoke test will exclude if no decoder
                                      notes='Decoder not implemented — latency only'),

    # ============================================================
    # DETECTION — VOC (Pascal VOC 2007)
    # ============================================================
    'yolov3_voc_tf':               m('detection', 'voc', decoder='yolo_voc',
                                      input_size=416, scale=[1/255]*3, ch='RGB'),
    'refinedet_VOC_tf':            m('detection', 'voc', decoder='refinedet',
                                      input_size=480, mean=[104, 117, 123],
                                      ch='BGR', enabled=False,
                                      notes='RefineDet 2-stage decoder not impl'),
    'refinedet_pruned_0_8':        m('detection', 'voc', decoder='refinedet',
                                      input_size=480, mean=[104, 117, 123],
                                      ch='BGR', enabled=False,
                                      notes='20% pruned RefineDet'),
    'refinedet_pruned_0_92':       m('detection', 'voc', decoder='refinedet',
                                      input_size=480, mean=[104, 117, 123],
                                      ch='BGR', enabled=False,
                                      notes='8% pruned RefineDet'),
    'refinedet_pruned_0_96':       m('detection', 'voc', decoder='refinedet',
                                      input_size=480, mean=[104, 117, 123],
                                      ch='BGR', enabled=False,
                                      notes='4% pruned RefineDet'),
    'mobilenet_edge_2_75_pt':      m('classification', 'imagenet',
                                      mean=IM_MEAN_RGB_PT, scale=IM_SCALE_PT,
                                      ch='RGB'),

    # ============================================================
    # FACE / SPECIAL — small models, may or may not work as detection
    # ============================================================
    'face_mask_detection_pt':      m('detection', 'voc', decoder=None,
                                      input_size=512, ch='BGR',
                                      enabled=True,
                                      notes='Custom face-mask model; no standard decoder'),
    'densebox_320_320':            m('detection', 'face', decoder=None,
                                      input_size=320, mean=[128, 128, 128],
                                      scale=[1.0]*3, ch='BGR',
                                      enabled=True,
                                      notes='Densebox face detection; no standard decoder'),
    'densebox_640_360':            m('detection', 'face', decoder=None,
                                      input_size=640, mean=[128, 128, 128],
                                      scale=[1.0]*3, ch='BGR',
                                      enabled=True,
                                      notes='Densebox face detection; no standard decoder'),
}

print(f"Catalogue: {len(CATALOG)} entries")
print(f"  Classification: {sum(1 for v in CATALOG.values() if v['category']=='classification')}")
print(f"  Detection COCO: {sum(1 for v in CATALOG.values() if v['dataset']=='coco')}")
print(f"  Detection VOC:  {sum(1 for v in CATALOG.values() if v['dataset']=='voc')}")
print(f"  Detection face: {sum(1 for v in CATALOG.values() if v['dataset']=='face')}")
print(f"  Enabled:        {sum(1 for v in CATALOG.values() if v['enabled'])}")
print(f"  Non-commercial: {sum(1 for v in CATALOG.values() if v['non_commercial'])}")
print()
print(f"Note: some entries have enabled=False because their decoder isn't implemented yet.")
print(f"Set enabled=True to include them in the latency/power benchmark anyway.")


## 4. Prerequisite check (no downloads)

This notebook **does not download anything**. The earlier in-notebook auto-download corrupted a 256 GB SD card under sustained writes; the data is now staged on the host PC instead.

**Before running this notebook**, run on your laptop:

```bash
cd ~/Documents/Girona_Masters/Thesis/KriaKv260_Model_Compiler
bash scripts/host/04_stage_benchmark.sh
bash scripts/host/05_sync_benchmark_to_kria.sh ubuntu@<this-kria-ip>
```

The cell below verifies everything is in place and reports clear errors if not.


In [ ]:
# Prerequisite check — verifies data was staged via the host-side scripts.
# This notebook does NO downloads. If you see errors below, run on your laptop:
#   bash scripts/host/04_stage_benchmark.sh
#   bash scripts/host/05_sync_benchmark_to_kria.sh ubuntu@<this-kria-ip>

from pathlib import Path

missing = []

if not MODELS_DIR.exists():
    missing.append(f"models root: {MODELS_DIR}")
else:
    n_xmodels = sum(1 for _ in MODELS_DIR.rglob("*.xmodel"))
    print(f"  Models:      {n_xmodels} xmodel files in {MODELS_DIR}")
    if n_xmodels == 0:
        missing.append(f"no xmodels under {MODELS_DIR}")

if not IMAGENET_IMAGES.exists() or not IMAGENET_LABELS.exists():
    missing.append(f"ImageNet: {IMAGENET_IMAGES.parent} (need both images/ and labels.txt)")
else:
    n_imgs = sum(1 for _ in IMAGENET_IMAGES.iterdir())
    n_labels = sum(1 for _ in IMAGENET_LABELS.open())
    print(f"  ImageNet:    {n_imgs} images, {n_labels} labels in {IMAGENET_IMAGES.parent}")

if not COCO_IMG_DIR.exists() or not COCO_ANN_FILE.exists():
    missing.append(f"COCO: {COCO_FULL_DIR}")
else:
    n_coco = sum(1 for _ in COCO_IMG_DIR.iterdir())
    print(f"  COCO:        {n_coco} images, annotations present at {COCO_ANN_FILE.name}")

if not VOC_IMG_DIR.exists() or not VOC_SPLIT_FILE.exists():
    missing.append(f"VOC2007 test: {VOC_DIR}")
else:
    n_voc = sum(1 for _ in VOC_IMG_DIR.iterdir())
    n_split = sum(1 for _ in VOC_SPLIT_FILE.open())
    print(f"  VOC2007:     {n_voc} images, {n_split} in test split")

iv2_json = DATASET_DIR / "imagenet_class_index.json"
if not iv2_json.exists():
    print(f"  [warn] {iv2_json.name} not found — name-based ImageNet labels won't resolve")
else:
    print(f"  Class index: {iv2_json.name} present")

if missing:
    print()
    print("=" * 70)
    print("MISSING PREREQUISITES")
    print("=" * 70)
    for m in missing:
        print(f"  - {m}")
    print()
    print("Run on your laptop (not on the Kria):")
    print("  bash scripts/host/04_stage_benchmark.sh")
    print("  bash scripts/host/05_sync_benchmark_to_kria.sh ubuntu@<this-kria-ip>")
    print()
    print("Then re-run this cell.")
    raise RuntimeError("Benchmark data not staged. See message above.")

print("\n  All prerequisites present. Proceed to the next cell.")


## 5. Sanity check — USB autosuspend off

Same as the 2.5 notebook. The BRIO+KV260 stack silently caps at 15 FPS if
autosuspend is on for the camera or any parent hub.


In [ ]:
def assert_autosuspend_off():
    bad = []
    candidates = ["2-1.2", "2-1", "usb2"]
    for d in candidates:
        p = Path(f"/sys/bus/usb/devices/{d}/power/control")
        if p.exists() and p.read_text().strip() != "on":
            bad.append(d)
    if bad:
        print(f"USB autosuspend ON for {bad}.")
        print("Fix from a board terminal:")
        for d in bad:
            print(f"  echo on | sudo tee /sys/bus/usb/devices/{d}/power/control")
        print()
        print("The camera-based benchmark will be unreliable until this is fixed.")
        print("Latency / accuracy / mAP cells will still work.")
    else:
        print("USB autosuspend: OK (all hubs 'on')")

assert_autosuspend_off()


## 6. Power monitor

Same code as the 2.5 notebook — reads hwmon sysfs to track board power. The
hwmon paths are KV260 hardware properties, unchanged across VAI versions.


In [ ]:
def _power_files():
    return sorted(glob.glob('/sys/class/hwmon/hwmon*/power*_input'))

def _vi_pairs():
    pairs = []
    for hd in sorted(glob.glob('/sys/class/hwmon/hwmon*')):
        for vf in sorted(glob.glob(f'{hd}/in*_input')):
            mt = re.search(r'in(\d+)_input', vf)
            if mt:
                cf = f'{hd}/curr{mt.group(1)}_input'
                if os.path.exists(cf):
                    pairs.append((vf, cf))
    return pairs

POWER_FILES = _power_files()
VI_PAIRS    = _vi_pairs() if not POWER_FILES else []

def read_power_w():
    total = 0.0
    for p in POWER_FILES:
        try:
            with open(p) as f: total += int(f.read()) / 1e6
        except Exception: pass
    for vf, cf in VI_PAIRS:
        try:
            with open(vf) as f: v = int(f.read())
            with open(cf) as f: i = int(f.read())
            total += (v * i) / 1e6
        except Exception: pass
    return total

print(f"Sensors: {len(POWER_FILES)} power_input + {len(VI_PAIRS)} V/I pairs")
print(f"Idle power: {read_power_w():.2f} W")


## 7. File helpers + preprocessing

`find_xmodel_for(name)` looks for the xmodel that matches the catalogue key.
Necessary because the Xilinx tar.gz files extract to dirs whose names may
not exactly match the catalogue key (e.g., `inception_v3_tf` extracts to
`inception_v3_tf/inception_v3_tf.xmodel`, but some pruned variants drop
the prefix).


In [ ]:
def find_xmodel_for(name, cfg):
    cat_dir = MODELS_DIR / cfg['category']
    if not cat_dir.exists():
        return None
    # Try direct match first
    direct = cat_dir / name / f"{name}.xmodel"
    if direct.exists():
        return direct
    # Then any xmodel under a directory whose name contains the catalogue key
    for p in cat_dir.rglob('*.xmodel'):
        if name.lower() in p.parent.name.lower() or name.lower() in p.name.lower():
            return p
    return None

def preprocess_for_model(img_path_or_array, cfg, in_shape):
    h, w = in_shape[1], in_shape[2]
    if isinstance(img_path_or_array, (str, Path)):
        img = Image.open(img_path_or_array).convert('RGB').resize((w, h), Image.BILINEAR)
        arr = np.array(img, dtype=np.float32)    # HWC, RGB
    else:
        # already an array (BGR from cv2)
        arr = cv2.resize(img_path_or_array, (w, h),
                          interpolation=cv2.INTER_LINEAR).astype(np.float32)
        # convert to RGB if the model expects RGB
        if cfg.get('ch', 'BGR') == 'RGB':
            arr = arr[..., ::-1].copy()
    if cfg.get('ch') == 'BGR' and isinstance(img_path_or_array, (str, Path)):
        arr = arr[..., ::-1].copy()              # RGB -> BGR
    mean = cfg.get('mean')
    scale = cfg.get('scale')
    if mean is not None:
        arr = arr - np.array(mean, dtype=np.float32)
    if scale is not None:
        arr = arr * np.array(scale, dtype=np.float32)
    elif mean is None:
        arr = arr / 255.0
    return arr

def pick_camera_preset(in_shape):
    h, w = in_shape[1], in_shape[2]
    for cw, ch, cf in CAMERA_PRESETS:
        if cw >= w and ch >= h:
            return cw, ch, cf
    return CAMERA_PRESETS[-1]


## 8. Dataset loaders

Three loaders: ImageNet sample (Top-1), COCO val2017 (mAP), VOC2007 test
(mAP). ImageNet loader is identical to the 2.5 notebook. COCO loader is
also identical. VOC loader is new (XML annotations + per-image text
files).


In [ ]:
# ─── ImageNet sample loader ───────────────────────────────────────────
def _normalize(s):
    return s.lower().strip().replace(' ', '_')

def _load_imagenet_name_to_idx():
    for p in [ROOT/"imagenet_class_index.json",
              DATASET_DIR/"imagenet_class_index.json",
              DATASET_DIR/"imagenet_sample"/"imagenet_class_index.json"]:
        if p.exists():
            data = json.loads(p.read_text())
            mapping = {}
            for idx, e in data.items():
                name = e[1] if isinstance(e, list) else e
                key = _normalize(name)
                mapping.setdefault(key, int(idx))
            return mapping, p
    return None, None

NAME_TO_IDX, _NF = _load_imagenet_name_to_idx()
if NAME_TO_IDX:
    print(f"ImageNet name-to-idx: {len(NAME_TO_IDX)} from {_NF.name}")
else:
    print("No imagenet_class_index.json — name-based labels won't work")

def load_imagenet_dataset(images_dir, labels_file, max_n=None):
    if not Path(labels_file).exists() or not Path(images_dir).exists():
        return []
    pairs, n_unknown = [], 0
    for line in Path(labels_file).read_text().strip().splitlines():
        parts = line.strip().split()
        if len(parts) < 2: continue
        fname = label = None
        if parts[-1].lstrip('-').isdigit() and not parts[0].lstrip('-').isdigit():
            fname, label = parts[0], int(parts[-1])
        elif parts[0].lstrip('-').isdigit():
            fname, label = parts[1], int(parts[0])
        elif NAME_TO_IDX is not None:
            fname = parts[0]
            key = _normalize(' '.join(parts[1:]))
            if key in NAME_TO_IDX: label = NAME_TO_IDX[key]
            else: n_unknown += 1; continue
        else: continue
        p = Path(images_dir) / fname
        if p.exists():
            pairs.append((p, label))
        if max_n and len(pairs) >= max_n:
            break
    if n_unknown:
        print(f"  warning: {n_unknown} labels not in name-to-index mapping")
    return pairs


# ─── COCO val2017 loader (identical to 2.5 notebook) ─────────────────
def load_coco_gt(ann_path, img_dir, max_n=None):
    data = json.loads(Path(ann_path).read_text())
    cats = sorted(data['categories'], key=lambda c: c['id'])
    cat_id_to_idx = {c['id']: i for i, c in enumerate(cats)}
    idx_to_cat_id = {i: c['id'] for i, c in enumerate(cats)}
    cat_names = [c['name'] for c in cats]
    img_meta = {im['id']: im for im in data['images']}
    gt = {}
    for ann in data['annotations']:
        if ann.get('iscrowd', 0): continue
        iid = ann['image_id']
        if iid not in img_meta: continue
        x, y, w, h = ann['bbox']
        gt.setdefault(iid, []).append({
            'bbox': [x, y, x+w, y+h],
            'cat_idx': cat_id_to_idx[ann['category_id']],
        })
    available = []
    for iid, im in sorted(img_meta.items()):
        p = Path(img_dir) / im['file_name']
        if p.exists():
            available.append({'id': iid, 'file_name': im['file_name'],
                              'width': im['width'], 'height': im['height'],
                              'path': p})
        if max_n and len(available) >= max_n:
            break
    return available, gt, cat_id_to_idx, idx_to_cat_id, cat_names


# ─── VOC2007 test loader (NEW) ────────────────────────────────────────
VOC_CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor',
]
VOC_NAME_TO_IDX = {n: i for i, n in enumerate(VOC_CLASSES)}

def load_voc_gt(voc_dir, ann_dir, split_file, max_n=None):
    '''Returns (images_list, gt_by_imageid, idx_to_name).
    
    gt: dict[image_id] -> [{'bbox': [x1,y1,x2,y2], 'cat_idx': int}]
    '''
    if not split_file.exists():
        return [], {}, VOC_CLASSES
    import xml.etree.ElementTree as ET
    img_ids = split_file.read_text().strip().splitlines()[:max_n] if max_n else \
              split_file.read_text().strip().splitlines()
    
    images, gt = [], {}
    for img_id in img_ids:
        img_path = voc_dir / f"{img_id}.jpg"
        ann_path = ann_dir / f"{img_id}.xml"
        if not img_path.exists() or not ann_path.exists():
            continue
        # Parse XML annotation
        try:
            tree = ET.parse(ann_path)
            root = tree.getroot()
            size = root.find('size')
            W = int(size.find('width').text)
            H = int(size.find('height').text)
            objs = []
            for obj in root.findall('object'):
                difficult = int(obj.find('difficult').text or '0')
                if difficult: continue   # standard PASCAL VOC eval skips difficult
                name = obj.find('name').text
                if name not in VOC_NAME_TO_IDX: continue
                bb = obj.find('bndbox')
                x1 = float(bb.find('xmin').text)
                y1 = float(bb.find('ymin').text)
                x2 = float(bb.find('xmax').text)
                y2 = float(bb.find('ymax').text)
                objs.append({'bbox': [x1, y1, x2, y2],
                              'cat_idx': VOC_NAME_TO_IDX[name]})
            images.append({'id': img_id, 'file_name': f"{img_id}.jpg",
                           'width': W, 'height': H, 'path': img_path})
            gt[img_id] = objs
        except Exception as e:
            print(f"  warning: parse failed for {img_id}: {e}")
    return images, gt, VOC_CLASSES


# Load all three
if IMAGENET_IMAGES.exists() and IMAGENET_LABELS.exists():
    ds_imagenet = load_imagenet_dataset(IMAGENET_IMAGES, IMAGENET_LABELS,
                                         max_n=N_ACC_IMAGES)
    print(f"ImageNet sample: {len(ds_imagenet)} images")
else:
    ds_imagenet = []
    print("ImageNet sample: not available")

if COCO_ANN_FILE.exists() and COCO_IMG_DIR.exists():
    coco_images, coco_gt, COCO_CAT_TO_IDX, COCO_IDX_TO_CAT, COCO_NAMES = \
        load_coco_gt(COCO_ANN_FILE, COCO_IMG_DIR, max_n=N_COCO_IMAGES)
    print(f"COCO val2017: {len(coco_images)} images, "
          f"{sum(len(v) for v in coco_gt.values())} annotations, "
          f"{len(COCO_NAMES)} categories")
else:
    coco_images, coco_gt, COCO_CAT_TO_IDX, COCO_NAMES = [], {}, {}, []
    print("COCO val2017: not available")

if VOC_IMG_DIR.exists() and VOC_SPLIT_FILE.exists():
    voc_images, voc_gt, VOC_NAMES = load_voc_gt(
        VOC_IMG_DIR, VOC_ANN_DIR, VOC_SPLIT_FILE, max_n=N_VOC_IMAGES)
    print(f"VOC2007 test: {len(voc_images)} images, "
          f"{sum(len(v) for v in voc_gt.values())} annotations, "
          f"{len(VOC_NAMES)} categories")
else:
    voc_images, voc_gt, VOC_NAMES = [], {}, VOC_CLASSES
    print("VOC2007 test: not available")


## 9. Load DPU overlay

Programs the FPGA fabric once. The expected fingerprint check warns if
something is off (wrong board, wrong VAI version, starter-kit still loaded).


In [ ]:
from pynq_dpu import DpuOverlay
overlay = DpuOverlay("dpu.bit")
print("DPU overlay loaded.")

# Quick fingerprint check via a trivial test xmodel could go here,
# but it requires loading an xmodel first. We'll do it as part of the
# smoke test below.


## 10. Decoders, NMS, IoU, mAP

Carried over from the 2.5 notebook:
- `decode_ssd_tf` for TF SSD models
- `decode_yolo` for YOLOv3/v4 (COCO)
- IoU, NMS, mAP computation

New here:
- `decode_yolo_voc` — same YOLO decoder but with `num_classes=20` for VOC
- VOC mAP wrapper (`compute_voc_map`) using the same 11-point /
  101-point interpolation


In [ ]:
# ─── IoU ──────────────────────────────────────────────────────────────
def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a[:, 0:1], a[:, 1:2], a[:, 2:3], a[:, 3:4]
    bx1, by1, bx2, by2 = b[:, 0],   b[:, 1],   b[:, 2],   b[:, 3]
    ix1 = np.maximum(ax1, bx1); iy1 = np.maximum(ay1, by1)
    ix2 = np.minimum(ax2, bx2); iy2 = np.minimum(ay2, by2)
    iw = np.clip(ix2 - ix1, 0, None); ih = np.clip(iy2 - iy1, 0, None)
    inter = iw * ih
    area_a = (ax2 - ax1) * (ay2 - ay1)
    area_b = (bx2 - bx1) * (by2 - by1)
    union = area_a + area_b - inter
    return np.where(union > 0, inter / union, 0.0)

def nms(boxes, scores, iou_thr):
    if len(boxes) == 0: return np.array([], dtype=int)
    order = scores.argsort()[::-1]
    keep = []
    while len(order) > 0:
        i = order[0]; keep.append(i)
        if len(order) == 1: break
        ious = iou_xyxy(boxes[i:i+1], boxes[order[1:]]).flatten()
        order = order[1:][ious <= iou_thr]
    return np.array(keep, dtype=int)

def _sigmoid(x):
    return np.where(x >= 0, 1.0/(1.0+np.exp(-x)), np.exp(x)/(1.0+np.exp(x)))


# ─── SSD-TF decoder (from 2.5 notebook) ──────────────────────────────
SSD_VARIANCES = np.array([10.0, 10.0, 5.0, 5.0], dtype=np.float32)

def generate_ssd_anchors(fm_shapes, ars_per_layer, scales_per_layer):
    anchors = []
    for (fh, fw), arlist, scales in zip(fm_shapes, ars_per_layer, scales_per_layer):
        s, s_next = scales
        for i in range(fh):
            for j in range(fw):
                cy = (i + 0.5) / fh
                cx = (j + 0.5) / fw
                for ar in arlist:
                    if ar == 1.0:
                        anchors.append([cy, cx, s, s])
                        s_extra = (s * s_next) ** 0.5
                        anchors.append([cy, cx, s_extra, s_extra])
                    else:
                        h = s / (ar ** 0.5); w = s * (ar ** 0.5)
                        anchors.append([cy, cx, h, w])
    return np.asarray(anchors, dtype=np.float32)

def ssd_mobilenet_anchors(input_size=300):
    fm_shapes = [(19,19),(10,10),(5,5),(3,3),(2,2),(1,1)]
    ars = [
        [1.0, 2.0, 0.5],
        [1.0, 2.0, 0.5, 3.0, 1.0/3.0],
        [1.0, 2.0, 0.5, 3.0, 1.0/3.0],
        [1.0, 2.0, 0.5, 3.0, 1.0/3.0],
        [1.0, 2.0, 0.5],
        [1.0, 2.0, 0.5],
    ]
    smin, smax = 0.2, 0.95
    n = len(fm_shapes)
    base = [smin + (smax-smin)*k/(n-1) for k in range(n)]
    base.append(1.0)
    scales = [(base[k], base[k+1]) for k in range(n)]
    return generate_ssd_anchors(fm_shapes, ars, scales)

def decode_ssd_tf(outputs, in_size, num_classes=91,
                   score_thresh=CONF_THRESH, nms_iou=NMS_IOU_THRESH,
                   max_det=MAX_DET_PER_IMG, class_remap=None):
    box_t = cls_t = None
    for arr in outputs:
        sh = arr.shape
        if sh[-1] == 4 or (len(sh) >= 2 and sh[-2] == 4):
            box_t = arr.reshape(-1, 4)
        else:
            cls_t = arr.reshape(-1, sh[-1])
    if box_t is None or cls_t is None: return []
    anchors = ssd_mobilenet_anchors(in_size)
    n = min(len(box_t), len(anchors))
    box_t, anchors, cls_t = box_t[:n], anchors[:n], cls_t[:n]
    ty = box_t[:, 0] / SSD_VARIANCES[0]; tx = box_t[:, 1] / SSD_VARIANCES[1]
    th = box_t[:, 2] / SSD_VARIANCES[2]; tw = box_t[:, 3] / SSD_VARIANCES[3]
    cy = ty*anchors[:,2] + anchors[:,0]; cx = tx*anchors[:,3] + anchors[:,1]
    h  = np.exp(th)*anchors[:,2]; w = np.exp(tw)*anchors[:,3]
    x1 = np.clip(cx-w/2,0,1); y1 = np.clip(cy-h/2,0,1)
    x2 = np.clip(cx+w/2,0,1); y2 = np.clip(cy+h/2,0,1)
    boxes = np.stack([x1, y1, x2, y2], axis=1)
    cls_t = cls_t - cls_t.max(axis=1, keepdims=True)
    probs = np.exp(cls_t); probs /= probs.sum(axis=1, keepdims=True)
    detections = []
    for c in range(1, probs.shape[1]):
        sc = probs[:, c]; mask = sc > score_thresh
        if not mask.any(): continue
        b = boxes[mask]; s = sc[mask]
        keep = nms(b, s, nms_iou)
        for k in keep:
            cls_out = class_remap.get(c, c-1) if class_remap else c-1
            detections.append((b[k], float(s[k]), cls_out))
    detections.sort(key=lambda d: -d[1])
    return detections[:max_det]


# ─── YOLO decoder (from 2.5 notebook, generalized for any class count) ─
YOLO_ANCHORS_416 = np.array([
    [[116, 90], [156, 198], [373, 326]],
    [[ 30, 61], [ 62,  45], [ 59, 119]],
    [[ 10, 13], [ 16,  30], [ 33,  23]],
], dtype=np.float32)

def decode_yolo(outputs, in_size, num_classes=80,
                score_thresh=CONF_THRESH, nms_iou=NMS_IOU_THRESH,
                max_det=MAX_DET_PER_IMG, anchors=YOLO_ANCHORS_416):
    if len(outputs) < 3: return []
    feats = []
    for arr in outputs:
        if arr.ndim == 4:
            if arr.shape[-1] == 3 * (5 + num_classes):
                feats.append(arr[0])
            elif arr.shape[1] == 3 * (5 + num_classes):
                feats.append(np.transpose(arr[0], (1,2,0)))
    if len(feats) < 3: return []
    feats.sort(key=lambda x: -x.shape[0])   # largest grid first
    feat_anchors = [anchors[2], anchors[1], anchors[0]]
    dets_by_cls = {}
    for feat, anc in zip(feats, feat_anchors):
        gh, gw = feat.shape[:2]
        feat = feat.reshape(gh, gw, 3, 5 + num_classes)
        gx, gy = np.meshgrid(np.arange(gw), np.arange(gh))
        bx = (_sigmoid(feat[..., 0]) + gx[..., None]) / gw
        by = (_sigmoid(feat[..., 1]) + gy[..., None]) / gh
        bw = np.exp(np.clip(feat[..., 2], -10, 10)) * anc[:, 0] / in_size
        bh = np.exp(np.clip(feat[..., 3], -10, 10)) * anc[:, 1] / in_size
        obj = _sigmoid(feat[..., 4]); cls = _sigmoid(feat[..., 5:])
        scores = obj[..., None] * cls
        bx = bx.flatten(); by = by.flatten()
        bw = bw.flatten(); bh = bh.flatten()
        x1 = np.clip(bx-bw/2, 0, 1); y1 = np.clip(by-bh/2, 0, 1)
        x2 = np.clip(bx+bw/2, 0, 1); y2 = np.clip(by+bh/2, 0, 1)
        boxes = np.stack([x1, y1, x2, y2], axis=1)
        scores = scores.reshape(-1, num_classes)
        for c in range(num_classes):
            sc = scores[:, c]; mask = sc > score_thresh
            if not mask.any(): continue
            dets_by_cls.setdefault(c, [])
            for b, s in zip(boxes[mask], sc[mask]):
                dets_by_cls[c].append((b, float(s)))
    out = []
    for c, entries in dets_by_cls.items():
        if not entries: continue
        b = np.array([e[0] for e in entries])
        s = np.array([e[1] for e in entries])
        keep = nms(b, s, nms_iou)
        for k in keep:
            out.append((b[k], float(s[k]), c))
    out.sort(key=lambda d: -d[1])
    return out[:max_det]

# VOC variant — same YOLO architecture, 20 classes
def decode_yolo_voc(outputs, in_size, **kw):
    return decode_yolo(outputs, in_size, num_classes=20, **kw)


# ─── mAP computation (from 2.5 notebook, generalized) ────────────────
def compute_ap(recalls, precisions):
    if len(recalls) == 0: return 0.0
    rec_levels = np.linspace(0, 1, 101)
    aps = []
    for r in rec_levels:
        precs = precisions[recalls >= r]
        aps.append(precs.max() if len(precs) > 0 else 0.0)
    return float(np.mean(aps))

def evaluate_per_image(predictions, ground_truth, iou_thr, img_meta):
    by_class_pr = {}
    by_class_gt = {}
    for img_id, dets in predictions.items():
        gt_list = ground_truth.get(img_id, [])
        meta = img_meta.get(img_id)
        if meta is None: continue
        W, H = meta['width'], meta['height']
        gt_by_class = {}
        for g in gt_list:
            gt_by_class.setdefault(g['cat_idx'], []).append(g['bbox'])
            by_class_gt[g['cat_idx']] = by_class_gt.get(g['cat_idx'], 0) + 1
        dets_sorted = sorted(dets, key=lambda d: -d[1])
        matched = {c: set() for c in gt_by_class}
        for box, score, cls in dets_sorted:
            bx = np.array([box[0]*W, box[1]*H, box[2]*W, box[3]*H], dtype=np.float32)
            tp = 0
            if cls in gt_by_class:
                gtboxes = np.array(gt_by_class[cls], dtype=np.float32)
                ious = iou_xyxy(bx[None,:], gtboxes).flatten()
                best = ious.argmax()
                if ious[best] >= iou_thr and best not in matched[cls]:
                    tp = 1
                    matched[cls].add(best)
            by_class_pr.setdefault(cls, []).append((score, tp))
    return by_class_pr, by_class_gt

def compute_map_generic(predictions, ground_truth, iou_thresholds, img_meta):
    results = {}
    for thr in iou_thresholds:
        by_pr, by_gt = evaluate_per_image(predictions, ground_truth, thr, img_meta)
        if not by_gt:
            results[round(thr, 3)] = 0.0; continue
        aps = []
        for cls, n_gt in by_gt.items():
            entries = sorted(by_pr.get(cls, []), key=lambda x: -x[0])
            if not entries:
                aps.append(0.0); continue
            tp = np.array([1 if e[1] else 0 for e in entries], dtype=np.float32)
            fp = 1 - tp
            tp_cum = np.cumsum(tp); fp_cum = np.cumsum(fp)
            recalls = tp_cum / n_gt
            precisions = tp_cum / (tp_cum + fp_cum + 1e-10)
            aps.append(compute_ap(recalls, precisions))
        results[round(thr, 3)] = float(np.mean(aps))
    results['mean'] = float(np.mean([v for k, v in results.items() if k != 'mean']))
    return results

# Convenience: COCO + VOC wrappers
COCO_IMG_META = {im['id']: im for im in coco_images} if coco_images else {}
VOC_IMG_META  = {im['id']: im for im in voc_images}  if voc_images  else {}

def compute_coco_map(predictions, iou_thresholds):
    return compute_map_generic(predictions, coco_gt, iou_thresholds, COCO_IMG_META)

def compute_voc_map(predictions, iou_thresholds=(0.5,)):
    return compute_map_generic(predictions, voc_gt, iou_thresholds, VOC_IMG_META)


# ─── Decoder dispatch ────────────────────────────────────────────────
def build_tf_ssd_class_remap():
    if not COCO_CAT_TO_IDX: return {}
    return {cid: contig for cid, contig in COCO_CAT_TO_IDX.items()}

TF_SSD_TO_COCO = build_tf_ssd_class_remap()

DECODERS = {
    'ssd_tf':       lambda outs, sz: decode_ssd_tf(outs, sz, class_remap=TF_SSD_TO_COCO),
    'yolo':         lambda outs, sz: decode_yolo(outs, sz, num_classes=80),
    'yolo_voc':     lambda outs, sz: decode_yolo_voc(outs, sz),
    'refinedet':    None,    # not implemented; smoke test will mark detection=0
    'efficientdet': None,
}

print(f"Decoders ready: {list(k for k,v in DECODERS.items() if v is not None)}")
print(f"Stubs (no accuracy):     {list(k for k,v in DECODERS.items() if v is None)}")


## 11. Smoke test gate ⭐

**New in this notebook.** Before running the full benchmark, every model
is loaded and run on exactly **one image**:

- **Pass**: model loads, inference returns valid output → include in benchmark
- **Pass-no-decoder**: model loads, inference works, but no decoder
  implemented → include for latency/power/FPS only; mAP/Top-1 skipped
- **Fail-load**: xmodel won't load → log error, exclude
- **Fail-infer**: model loads but execute_async raises → log error, exclude
- **Fail-decode**: decoder runs but produces zero detections after threshold
  → STILL INCLUDED (might be that the test image has no detectable objects)

The smoke test results go to `vai35_smoke_test.csv`. Re-run this cell to
refresh; the main benchmark cells will only use models that passed.

For detection models, the test image is the first COCO/VOC image. For
classification, it's the first ImageNet sample (random tensor if no
images available — load+infer is what matters, not the answer).


In [ ]:
def smoke_test_one(name, cfg):
    result = {
        'model': name, 'category': cfg['category'], 'dataset': cfg.get('dataset',''),
        'decoder': cfg.get('decoder',''),
        'xmodel_found': False, 'load_ok': False, 'infer_ok': False,
        'decode_ok': None, 'first_detection': '',
        'output_shapes': '', 'input_shape': '',
        'error': '',
    }
    xm = find_xmodel_for(name, cfg)
    if not xm:
        result['error'] = 'xmodel not found in Models_VAI35/'
        return result
    result['xmodel_found'] = True
    
    input_data = output_data = dpu = None
    try:
        overlay.load_model(str(xm))
        dpu = overlay.runner
        result['load_ok'] = True
        in_t = dpu.get_input_tensors()
        out_t = dpu.get_output_tensors()
        in_shape = tuple(in_t[0].dims)
        result['input_shape'] = 'x'.join(str(d) for d in in_shape)
        result['output_shapes'] = ' | '.join(
            'x'.join(str(d) for d in t.dims) for t in out_t)
        input_data = [np.empty(in_shape, dtype=np.float32, order='C')]
        output_data = [np.empty(tuple(t.dims), dtype=np.float32, order='C')
                        for t in out_t]
        
        # Pick a sample image
        sample_path = None
        if cfg['category'] == 'classification' and ds_imagenet:
            sample_path = ds_imagenet[0][0]
        elif cfg.get('dataset') == 'coco' and coco_images:
            sample_path = coco_images[0]['path']
        elif cfg.get('dataset') in ('voc', 'face') and voc_images:
            sample_path = voc_images[0]['path']
        elif coco_images:
            sample_path = coco_images[0]['path']
        
        if sample_path:
            input_data[0][0] = preprocess_for_model(sample_path, cfg, in_shape)
        else:
            input_data[0][:] = (np.random.rand(*in_shape).astype(np.float32))
        
        dpu.wait(dpu.execute_async(input_data, output_data))
        result['infer_ok'] = True
        
        # Try to decode
        decoder_name = cfg.get('decoder')
        if decoder_name == 'classification':
            # Classification "decode" = argmax succeeds
            flat = output_data[0].flatten()
            top = int(flat.argmax())
            result['decode_ok'] = True
            result['first_detection'] = f"top1={top}"
        elif decoder_name in DECODERS and DECODERS[decoder_name] is not None:
            outs = [o[0] for o in output_data]
            try:
                dets = DECODERS[decoder_name](outs, cfg['input_size'])
                result['decode_ok'] = True
                if dets:
                    b, s, c = dets[0]
                    result['first_detection'] = (
                        f"cls={c} score={s:.3f} "
                        f"box=[{b[0]:.2f},{b[1]:.2f},{b[2]:.2f},{b[3]:.2f}] "
                        f"(n_det={len(dets)})"
                    )
                else:
                    result['first_detection'] = '(no detections > threshold)'
            except Exception as e:
                result['decode_ok'] = False
                result['error'] = f"decoder failed: {type(e).__name__}: {e}"
        else:
            # No decoder — but inference works
            result['decode_ok'] = None
            result['first_detection'] = '(no decoder; will measure FPS only)'
        
    except Exception as e:
        result['error'] = f"{type(e).__name__}: {e}"
        traceback.print_exc()
    finally:
        try: del input_data, output_data, dpu
        except Exception: pass
        gc.collect()
        time.sleep(0.2)
    return result


SMOKE_FIELDS = ['model','category','dataset','decoder',
                 'xmodel_found','load_ok','infer_ok','decode_ok',
                 'first_detection','output_shapes','input_shape','error']

# Resume — keep existing smoke results unless force-rerun
done_smoke = {}
if SMOKE_CSV.exists():
    for r in csv.DictReader(open(SMOKE_CSV)):
        done_smoke[r['model']] = r
    print(f"Resume: {len(done_smoke)} models already smoke-tested")

new_csv = not SMOKE_CSV.exists()
print(f"\nRunning smoke test on {sum(1 for v in CATALOG.values() if v['enabled'])} enabled models...")
print(f"(Delete {SMOKE_CSV.name} to force re-test)\n")

with open(SMOKE_CSV, 'a', newline='') as fh:
    w = csv.DictWriter(fh, fieldnames=SMOKE_FIELDS)
    if new_csv: w.writeheader(); fh.flush()
    for i, (name, cfg) in enumerate(
            [(n, c) for n, c in CATALOG.items() if c['enabled']], 1):
        if name in done_smoke:
            r = done_smoke[name]
            tag = ('OK' if r['infer_ok'] in ('True','true','1') else 'FAIL')
            print(f"  [{i:>2}] {tag:<4s} {name}  (cached)")
            continue
        r = smoke_test_one(name, cfg)
        w.writerow({k: r.get(k, '') for k in SMOKE_FIELDS}); fh.flush()
        if r['infer_ok']:
            tag = ('OK' if r['decode_ok'] is not False else 'DECODE-FAIL')
            print(f"  [{i:>2}] {tag:<11s} {name}: {r['first_detection']}")
        else:
            print(f"  [{i:>2}] FAIL  {name}: {r['error']}")
        done_smoke[name] = r

print()
ok    = [n for n, r in done_smoke.items() if r['infer_ok'] in ('True','true','1', True)]
fail  = [n for n, r in done_smoke.items() if r['infer_ok'] not in ('True','true','1', True)]
print(f"Summary: {len(ok)} models will run in the full benchmark; {len(fail)} excluded")
print(f"Failed: {fail[:10]}{'...' if len(fail) > 10 else ''}")


## 12. Main unified benchmark loop

Same structure as the 2.5 notebook's main loop:

1. Load model, allocate buffers
2. Seed one image to warm up
3. Measure idle power (0.3 s)
4. Warm-up the DPU (`N_WARMUP` iterations)
5. Pure-DPU latency + load power (`N_LAT_ITER` iterations)
6. Accuracy: ImageNet Top-1 for classification, `N` images
7. Camera benchmark: BRIO MJPG capture, per-stage timing
8. Compute FPS/W

**Only iterates over models that passed the smoke test** (`infer_ok=True`).
Models without a decoder still get latency / power / FPS measurements;
their `accuracy_top1` field stays None.

Resume support: skip models already in `vai35_benchmark_results.csv`.


In [ ]:
def benchmark_one_model(name, cfg):
    xm = find_xmodel_for(name, cfg)
    result = {
        'model': name, 'category': cfg['category'], 'dataset': cfg.get('dataset',''),
        'decoder': cfg.get('decoder',''), 'input_shape': '',
        'xmodel': str(xm) if xm else '',
        'dpu_latency_mean_ms': None, 'dpu_latency_p50_ms': None,
        'dpu_latency_p99_ms': None, 'dpu_fps': None,
        'power_idle_w': None, 'power_load_w': None,
        'accuracy_top1': None, 'n_acc_images': 0,
        'cam_resolution': '', 'cam_target_fps': None,
        'cam_fps': None, 'cam_frames': 0, 'cam_dropped': 0,
        'cam_capture_ms': None, 'cam_preprocess_ms': None, 'cam_inference_ms': None,
        'cam_limited': None,
        'fps_per_w': None,
        'error': '',
    }
    if not xm:
        result['error'] = 'xmodel not found'
        return result
    
    input_data = output_data = dpu = cap = None
    try:
        overlay.load_model(str(xm))
        dpu = overlay.runner
        in_t = dpu.get_input_tensors()
        out_t = dpu.get_output_tensors()
        in_shape = tuple(in_t[0].dims)
        result['input_shape'] = 'x'.join(str(d) for d in in_shape)
        input_data = [np.empty(in_shape, dtype=np.float32, order='C')]
        output_data = [np.empty(tuple(t.dims), dtype=np.float32, order='C')
                        for t in out_t]
        
        # Seed
        sample = None
        if cfg['category'] == 'classification' and ds_imagenet:
            sample = ds_imagenet[0][0]
        elif cfg.get('dataset') == 'coco' and coco_images:
            sample = coco_images[0]['path']
        elif cfg.get('dataset') in ('voc', 'face') and voc_images:
            sample = voc_images[0]['path']
        elif coco_images:
            sample = coco_images[0]['path']
        if sample:
            input_data[0][0] = preprocess_for_model(sample, cfg, in_shape)
        else:
            input_data[0][:] = (np.random.rand(*in_shape).astype(np.float32))
        
        # Idle power
        idle = [read_power_w()]
        time.sleep(0.3)
        idle.append(read_power_w())
        result['power_idle_w'] = round(float(np.mean(idle)), 3)
        
        # Warm-up
        for _ in range(N_WARMUP):
            dpu.wait(dpu.execute_async(input_data, output_data))
        
        # Pure-DPU latency + load power
        latencies, powers = [], []
        for _ in range(N_LAT_ITER):
            t0 = time.perf_counter()
            dpu.wait(dpu.execute_async(input_data, output_data))
            latencies.append((time.perf_counter() - t0) * 1000.0)
            powers.append(read_power_w())
        result['dpu_latency_mean_ms'] = round(float(np.mean(latencies)), 3)
        result['dpu_latency_p50_ms']  = round(float(np.percentile(latencies, 50)), 3)
        result['dpu_latency_p99_ms']  = round(float(np.percentile(latencies, 99)), 3)
        result['power_load_w']        = round(float(np.mean(powers)), 3)
        result['dpu_fps'] = round(1000.0 / result['dpu_latency_mean_ms'], 2)
        
        # Top-1 accuracy (classification only)
        if cfg['category'] == 'classification' and ds_imagenet:
            ds = ds_imagenet[:N_ACC_IMAGES]
            correct = 0
            for img_path, label in ds:
                input_data[0][0] = preprocess_for_model(img_path, cfg, in_shape)
                dpu.wait(dpu.execute_async(input_data, output_data))
                flat = output_data[0].flatten()
                pred = int(flat.argmax())
                if flat.size == 1001: pred -= 1
                correct += int(pred == label)
            result['accuracy_top1'] = round(correct / len(ds), 4) if ds else None
            result['n_acc_images'] = len(ds)
        
        # Camera benchmark
        cw, ch, cf = pick_camera_preset(in_shape)
        result['cam_resolution'] = f'{cw}x{ch}'
        result['cam_target_fps'] = cf
        cap = cv2.VideoCapture(CAMERA_DEVICE, cv2.CAP_V4L2)
        if not cap.isOpened():
            raise RuntimeError("camera open failed")
        cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*CAMERA_FOURCC))
        cap.set(cv2.CAP_PROP_FRAME_WIDTH,  cw)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, ch)
        cap.set(cv2.CAP_PROP_FPS, cf)
        try: cap.set(cv2.CAP_PROP_BUFFERSIZE, CAMERA_BUFFER)
        except Exception: pass
        
        for _ in range(5):
            ok, frame = cap.read()
            if not ok: continue
            input_data[0][0] = preprocess_for_model(frame, cfg, in_shape)
            dpu.wait(dpu.execute_async(input_data, output_data))
        
        n, n_drop = 0, 0
        t_cap = t_pre = t_inf = 0.0
        t0 = time.perf_counter()
        while time.perf_counter() - t0 < CAMERA_DURATION:
            a = time.perf_counter()
            ok, frame = cap.read()
            b = time.perf_counter()
            if not ok or frame is None:
                n_drop += 1; continue
            input_data[0][0] = preprocess_for_model(frame, cfg, in_shape)
            c = time.perf_counter()
            dpu.wait(dpu.execute_async(input_data, output_data))
            d = time.perf_counter()
            t_cap += b-a; t_pre += c-b; t_inf += d-c
            n += 1
        elapsed = time.perf_counter() - t0
        
        if n > 0:
            result['cam_frames']        = n
            result['cam_dropped']       = n_drop
            result['cam_fps']           = round(n / elapsed, 2)
            result['cam_capture_ms']    = round(t_cap/n*1000, 2)
            result['cam_preprocess_ms'] = round(t_pre/n*1000, 2)
            result['cam_inference_ms']  = round(t_inf/n*1000, 2)
            result['cam_limited']       = bool(result['cam_fps'] >= 0.95 * cf)
            if result['power_load_w'] and result['power_load_w'] > 0:
                result['fps_per_w'] = round(result['cam_fps']/result['power_load_w'], 3)
    
    except Exception as e:
        result['error'] = f"{type(e).__name__}: {e}"
        traceback.print_exc()
    finally:
        if cap is not None:
            try: cap.release()
            except Exception: pass
        try: del input_data, output_data, dpu
        except Exception: pass
        gc.collect()
    return result


# Pick the working models from smoke test
working_models = []
for name, cfg in CATALOG.items():
    if not cfg['enabled']: continue
    smoke = done_smoke.get(name)
    if smoke and smoke.get('infer_ok') in ('True','true','1', True):
        working_models.append((name, cfg))
print(f"Will benchmark {len(working_models)} models that passed smoke test")

FIELDS = [
    'model','category','dataset','decoder','input_shape','xmodel',
    'dpu_latency_mean_ms','dpu_latency_p50_ms','dpu_latency_p99_ms','dpu_fps',
    'power_idle_w','power_load_w',
    'accuracy_top1','n_acc_images',
    'cam_resolution','cam_target_fps','cam_fps','cam_frames','cam_dropped',
    'cam_capture_ms','cam_preprocess_ms','cam_inference_ms','cam_limited',
    'fps_per_w','error',
]

done = set()
if RESULTS_CSV.exists():
    for r in csv.DictReader(open(RESULTS_CSV)):
        done.add(r['model'])
    print(f"Resume: {len(done)} models already in CSV, skipping those")

new_csv = not RESULTS_CSV.exists()
with open(RESULTS_CSV, 'a', newline='') as fh:
    w = csv.DictWriter(fh, fieldnames=FIELDS)
    if new_csv: w.writeheader(); fh.flush()
    for i, (name, cfg) in enumerate(working_models, 1):
        if name in done:
            print(f"[{i:>2}/{len(working_models)}] skip {name}")
            continue
        print(f"[{i:>2}/{len(working_models)}] run  {name}")
        r = benchmark_one_model(name, cfg)
        w.writerow({k: r.get(k, '') for k in FIELDS}); fh.flush()
        if r['error']:
            print(f"     ERR  {r['error']}")
        else:
            print(f"     dpu_fps={r['dpu_fps']}  cam_fps={r['cam_fps']}  "
                  f"pwr={r['power_load_w']}W  fps/W={r['fps_per_w']}  "
                  f"top1={r['accuracy_top1']}")
        time.sleep(0.3)

print(f"\nDone. CSV at {RESULTS_CSV}")


## 13. COCO mAP loop

For models in `CATALOG` where `dataset == 'coco'` and `decoder` is in
`{'ssd_tf', 'yolo'}`, computes mAP@0.5 and mAP@0.5:0.95 against COCO
val2017. Same evaluation methodology as the 2.5 notebook (pure NumPy,
101-point AP, COCO-style IoU threshold range).


In [ ]:
def evaluate_one_detection_model(name, cfg, dataset, decoder_fn):
    '''Generic detection evaluator. `dataset` is the list of image
    dicts; `decoder_fn` returns dets per image.'''
    result = {
        'model': name, 'evaluable': True, 'skip_reason': '',
        'n_images': 0, 'n_total_dets': 0,
        'map_50': None, 'map_50_95': None,
        'eval_time_s': None, 'error': '',
    }
    xm = find_xmodel_for(name, cfg)
    if not xm:
        result['error'] = 'xmodel not found'; return result
    try:
        overlay.load_model(str(xm))
        dpu = overlay.runner
        in_t = dpu.get_input_tensors()
        out_t = dpu.get_output_tensors()
        in_shape = tuple(in_t[0].dims)
        input_data = [np.empty(in_shape, dtype=np.float32, order='C')]
        output_data = [np.empty(tuple(t.dims), dtype=np.float32, order='C')
                        for t in out_t]
        for _ in range(3):
            dpu.wait(dpu.execute_async(input_data, output_data))
        
        predictions = {}
        t0 = time.perf_counter()
        for k, img_meta in enumerate(dataset):
            try:
                input_data[0][0] = preprocess_for_model(img_meta['path'], cfg, in_shape)
            except Exception:
                continue
            dpu.wait(dpu.execute_async(input_data, output_data))
            outs = [o[0] for o in output_data]
            try:
                dets = decoder_fn(outs, cfg['input_size'])
            except Exception:
                dets = []
            predictions[img_meta['id']] = dets
            if (k+1) % 500 == 0:
                print(f"    {k+1}/{len(dataset)} images")
        elapsed = time.perf_counter() - t0
        result['eval_time_s']  = round(elapsed, 1)
        result['n_images']     = len(predictions)
        result['n_total_dets'] = sum(len(v) for v in predictions.values())
        return predictions, result
    except Exception as e:
        result['error'] = f"{type(e).__name__}: {e}"
        traceback.print_exc()
        return {}, result
    finally:
        try: del input_data, output_data, dpu
        except Exception: pass
        gc.collect()
        time.sleep(0.2)


MAP_FIELDS = ['model','dataset','decoder','evaluable','skip_reason',
              'n_images','n_total_dets','map_50','map_50_95','eval_time_s','error']

# COCO models
coco_models = [(n, c) for n, c in CATALOG.items()
                if c['enabled'] and c.get('dataset') == 'coco']
print(f"COCO mAP: {len(coco_models)} models to evaluate")

done_coco = set()
if COCO_MAP_CSV.exists():
    for r in csv.DictReader(open(COCO_MAP_CSV)):
        done_coco.add(r['model'])
    print(f"Resume: {len(done_coco)} already in CSV")

new_csv = not COCO_MAP_CSV.exists()
if coco_images:
    with open(COCO_MAP_CSV, 'a', newline='') as fh:
        w = csv.DictWriter(fh, fieldnames=MAP_FIELDS)
        if new_csv: w.writeheader(); fh.flush()
        for i, (name, cfg) in enumerate(coco_models, 1):
            if name in done_coco:
                print(f"[{i:>2}/{len(coco_models)}] skip {name}")
                continue
            decoder_name = cfg.get('decoder')
            row = {'model': name, 'dataset': 'coco', 'decoder': decoder_name or '',
                    'evaluable': False, 'skip_reason': '',
                    'n_images': 0, 'n_total_dets': 0,
                    'map_50': '', 'map_50_95': '', 'eval_time_s': '', 'error': ''}
            if decoder_name not in DECODERS or DECODERS[decoder_name] is None:
                row['skip_reason'] = f"decoder {decoder_name!r} not implemented"
                w.writerow(row); fh.flush()
                print(f"[{i:>2}/{len(coco_models)}] skip {name}: {row['skip_reason']}")
                continue
            print(f"[{i:>2}/{len(coco_models)}] eval {name} ...")
            preds, eval_r = evaluate_one_detection_model(
                name, cfg, coco_images, DECODERS[decoder_name])
            if eval_r['error']:
                row.update(eval_r); row['evaluable'] = True
                w.writerow({k: row.get(k, '') for k in MAP_FIELDS}); fh.flush()
                print(f"     ERROR: {eval_r['error']}")
                continue
            m50 = compute_coco_map(preds, [0.5])[0.5]
            thr_range = np.arange(0.5, 1.0, 0.05).tolist()
            m_all = compute_coco_map(preds, thr_range)
            row.update({
                'evaluable': True,
                'n_images': eval_r['n_images'],
                'n_total_dets': eval_r['n_total_dets'],
                'eval_time_s': eval_r['eval_time_s'],
                'map_50': round(m50, 4),
                'map_50_95': round(m_all['mean'], 4),
            })
            w.writerow({k: row.get(k, '') for k in MAP_FIELDS}); fh.flush()
            print(f"     mAP@0.5={row['map_50']} mAP@0.5:0.95={row['map_50_95']} "
                  f"({eval_r['n_images']} imgs, {eval_r['eval_time_s']}s)")
else:
    print("COCO images not loaded; skipping mAP loop.")


## 14. VOC mAP loop

Same pattern as the COCO loop, but with VOC2007 test annotations and
the `yolo_voc` decoder (or anything else producing 20-class output in
VOC class order).


In [ ]:
voc_models = [(n, c) for n, c in CATALOG.items()
                if c['enabled'] and c.get('dataset') == 'voc']
print(f"VOC mAP: {len(voc_models)} models to evaluate")

done_voc = set()
if VOC_MAP_CSV.exists():
    for r in csv.DictReader(open(VOC_MAP_CSV)):
        done_voc.add(r['model'])
    print(f"Resume: {len(done_voc)} already in CSV")

new_csv = not VOC_MAP_CSV.exists()
if voc_images:
    with open(VOC_MAP_CSV, 'a', newline='') as fh:
        w = csv.DictWriter(fh, fieldnames=MAP_FIELDS)
        if new_csv: w.writeheader(); fh.flush()
        for i, (name, cfg) in enumerate(voc_models, 1):
            if name in done_voc:
                print(f"[{i:>2}/{len(voc_models)}] skip {name}")
                continue
            decoder_name = cfg.get('decoder')
            row = {'model': name, 'dataset': 'voc', 'decoder': decoder_name or '',
                    'evaluable': False, 'skip_reason': '',
                    'n_images': 0, 'n_total_dets': 0,
                    'map_50': '', 'map_50_95': '', 'eval_time_s': '', 'error': ''}
            if decoder_name not in DECODERS or DECODERS[decoder_name] is None:
                row['skip_reason'] = f"decoder {decoder_name!r} not implemented"
                w.writerow(row); fh.flush()
                print(f"[{i:>2}/{len(voc_models)}] skip {name}: {row['skip_reason']}")
                continue
            print(f"[{i:>2}/{len(voc_models)}] eval {name} ...")
            preds, eval_r = evaluate_one_detection_model(
                name, cfg, voc_images, DECODERS[decoder_name])
            if eval_r['error']:
                row.update(eval_r); row['evaluable'] = True
                w.writerow({k: row.get(k, '') for k in MAP_FIELDS}); fh.flush()
                print(f"     ERROR: {eval_r['error']}")
                continue
            m50 = compute_voc_map(preds, [0.5])[0.5]
            # VOC traditionally only reports mAP@0.5; we still compute 0.5:0.95
            thr_range = np.arange(0.5, 1.0, 0.05).tolist()
            m_all = compute_voc_map(preds, thr_range)
            row.update({
                'evaluable': True,
                'n_images': eval_r['n_images'],
                'n_total_dets': eval_r['n_total_dets'],
                'eval_time_s': eval_r['eval_time_s'],
                'map_50': round(m50, 4),
                'map_50_95': round(m_all['mean'], 4),
            })
            w.writerow({k: row.get(k, '') for k in MAP_FIELDS}); fh.flush()
            print(f"     mAP@0.5={row['map_50']} mAP@0.5:0.95={row['map_50_95']} "
                  f"({eval_r['n_images']} imgs, {eval_r['eval_time_s']}s)")
else:
    print("VOC images not loaded; skipping mAP loop.")


## 15. Combined report

Reads all four CSVs (smoke test, main benchmark, COCO mAP, VOC mAP) and
emits a single markdown report covering the full benchmark.


In [ ]:
def fmt(v, fallback='-'):
    if v is None or v == '' or (isinstance(v, str) and not v.strip()):
        return fallback
    return str(v)

def to_float(v, default=None):
    try: return float(v)
    except (TypeError, ValueError): return default


def generate_combined_report():
    if not RESULTS_CSV.exists():
        print("Run the main benchmark first"); return
    rows = list(csv.DictReader(open(RESULTS_CSV)))
    coco_map_rows = list(csv.DictReader(open(COCO_MAP_CSV))) if COCO_MAP_CSV.exists() else []
    voc_map_rows  = list(csv.DictReader(open(VOC_MAP_CSV)))  if VOC_MAP_CSV.exists()  else []
    smoke_rows    = list(csv.DictReader(open(SMOKE_CSV)))    if SMOKE_CSV.exists()    else []
    
    coco_by_name  = {r['model']: r for r in coco_map_rows}
    voc_by_name   = {r['model']: r for r in voc_map_rows}
    smoke_by_name = {r['model']: r for r in smoke_rows}
    
    cls = [r for r in rows if r['category'] == 'classification']
    det = [r for r in rows if r['category'] == 'detection']
    err = [r for r in rows if r['error']]
    smoke_failed = [r for r in smoke_rows if r['infer_ok'] not in ('True','true','1', True)]
    
    out = []; P = lambda *a: out.append(' '.join(str(x) for x in a))
    
    # ── Header ──
    P("# KV260 Vitis AI 3.5 Benchmark Report")
    P("")
    P("Generated:", time.strftime('%Y-%m-%d %H:%M:%S'))
    P("")
    P("## Test Environment")
    P("")
    P("| Field | Value |")
    P("|---|---|")
    P("| Board | Kria KV260 |")
    P("| Stack | DPU-PYNQ design_contest_3.5 / Vitis AI 3.5 / DPUCZDX8G_ISA1_B4096 |")
    P("| DPU fingerprint | 0x101000056010407 (B4096 @ 300 MHz) |")
    P(f"| Camera | Logitech BRIO on /dev/video{CAMERA_DEVICE}, MJPG, BUFFERSIZE={CAMERA_BUFFER} |")
    P(f"| Cls dataset | ImageNet sample, up to {N_ACC_IMAGES} images for Top-1 |")
    P(f"| Det dataset (COCO) | COCO val2017 ({len(coco_images) if coco_images else 0} images for mAP) |")
    P(f"| Det dataset (VOC) | VOC2007 test ({len(voc_images) if voc_images else 0} images for mAP) |")
    P(f"| DPU latency iterations | {N_LAT_ITER} per model (after {N_WARMUP} warm-up) |")
    P(f"| Camera duration | {CAMERA_DURATION} s per model |")
    P("")
    P("**Note on model versions**: pre-compiled xmodels were downloaded from")
    P("AMD's VAI 3.0 model zoo (KV260 binaries). The DPU IP (B4096) is")
    P("unchanged between VAI 3.0 and VAI 3.5, so these binaries run correctly")
    P("on the VAI 3.5 runtime per [AMD's compatibility note](https://wiki.trenz-electronic.de/display/PD/Compilation+of+AI+3.0+models+for+Vitis+2023.2,+AI+3.5+SW,+AI+3.0+DPUCZDX8G).")
    P("")
    
    # ── Summary ──
    P("## Summary")
    P("")
    P("| Statistic | Value |")
    P("|---|---|")
    P(f"| Catalogue total | {len(CATALOG)} |")
    P(f"| Enabled in catalogue | {sum(1 for v in CATALOG.values() if v['enabled'])} |")
    P(f"| Smoke test passed | {sum(1 for r in smoke_rows if r['infer_ok'] in ('True','true','1', True))} |")
    P(f"| Smoke test failed | {len(smoke_failed)} |")
    P(f"| Fully benchmarked | {len(rows)} |")
    P(f"| Classification | {len(cls)} |")
    P(f"| Detection | {len(det)} |")
    P(f"| Detection w/ COCO mAP | {sum(1 for m in coco_map_rows if to_float(m.get('map_50')) is not None)} |")
    P(f"| Detection w/ VOC mAP | {sum(1 for m in voc_map_rows if to_float(m.get('map_50')) is not None)} |")
    P(f"| Errors in main loop | {len(err)} |")
    P("")
    
    # ── Smoke test failures ──
    if smoke_failed:
        P("## Smoke Test Failures")
        P("")
        P("These models failed to load or produce inference output. They were")
        P("excluded from the full benchmark. Common causes: corrupt download,")
        P("incompatible xmodel format, fingerprint mismatch.")
        P("")
        P("| Model | xmodel found | Load OK | Infer OK | Error |")
        P("|---|:-:|:-:|:-:|---|")
        for r in smoke_failed:
            P(f"| {r['model']} | {r['xmodel_found']} | {r['load_ok']} | "
              f"{r['infer_ok']} | `{r['error'][:60]}` |")
        P("")
    
    # ── Detection accuracy ──
    if coco_map_rows or voc_map_rows:
        P("## Detection Accuracy")
        P("")
        if coco_map_rows:
            P("### COCO val2017")
            P("")
            P(f"N={len(coco_images) if coco_images else 0} images, NMS IoU={NMS_IOU_THRESH}, "
              f"score>={CONF_THRESH}, max {MAX_DET_PER_IMG} det/image.")
            P("")
            P("| Model | Decoder | Status | mAP@0.5 | mAP@0.5:0.95 | n_imgs |")
            P("|---|---|---|---:|---:|---:|")
            for r in coco_map_rows:
                if not r['evaluable'] in ('True','true','1', True):
                    P(f"| {r['model']} | {r['decoder']} | SKIPPED | -- | -- | -- |")
                elif r['error']:
                    P(f"| {r['model']} | {r['decoder']} | ERROR | -- | -- | -- |")
                else:
                    P(f"| {r['model']} | {r['decoder']} | OK | "
                      f"{r['map_50']} | {r['map_50_95']} | {r['n_images']} |")
            P("")
        if voc_map_rows:
            P("### VOC2007 test")
            P("")
            P(f"N={len(voc_images) if voc_images else 0} images, NMS IoU={NMS_IOU_THRESH}, "
              f"score>={CONF_THRESH}, max {MAX_DET_PER_IMG} det/image.")
            P("")
            P("| Model | Decoder | Status | mAP@0.5 | mAP@0.5:0.95 | n_imgs |")
            P("|---|---|---|---:|---:|---:|")
            for r in voc_map_rows:
                if not r['evaluable'] in ('True','true','1', True):
                    P(f"| {r['model']} | {r['decoder']} | SKIPPED | -- | -- | -- |")
                elif r['error']:
                    P(f"| {r['model']} | {r['decoder']} | ERROR | -- | -- | -- |")
                else:
                    P(f"| {r['model']} | {r['decoder']} | OK | "
                      f"{r['map_50']} | {r['map_50_95']} | {r['n_images']} |")
            P("")
    
    # ── Top rankings ──
    def top(rs, key, n=10, reverse=True):
        return sorted([r for r in rs if to_float(r.get(key)) is not None and not r.get('error')],
                       key=lambda r: to_float(r[key]), reverse=reverse)[:n]
    
    P("## Top Rankings")
    P("")
    P("### Top 10 by Pure-DPU FPS")
    P("")
    P("| Rank | Model | Cat | Input | DPU FPS | Latency mean (ms) | Power (W) |")
    P("|---|---|---|---|---:|---:|---:|")
    for i, r in enumerate(top(rows, 'dpu_fps'), 1):
        P(f"| {i} | {r['model']} | {r['category'][:5]} | {r['input_shape']} | "
          f"{r['dpu_fps']} | {r['dpu_latency_mean_ms']} | {r['power_load_w']} |")
    P("")
    
    P("### Top 10 by Camera FPS")
    P("")
    P("| Rank | Model | Cat | Input | Cam res | Cam FPS | Cam-bound | Power (W) |")
    P("|---|---|---|---|---|---:|:---:|---:|")
    for i, r in enumerate(top(rows, 'cam_fps'), 1):
        cb = 'yes' if r['cam_limited'] in ('True','true','1') else 'no'
        P(f"| {i} | {r['model']} | {r['category'][:5]} | {r['input_shape']} | "
          f"{r['cam_resolution']} | {r['cam_fps']} | {cb} | {r['power_load_w']} |")
    P("")
    
    P("### Top 10 by FPS/W")
    P("")
    P("| Rank | Model | Cat | Cam FPS | Power (W) | FPS/W |")
    P("|---|---|---|---:|---:|---:|")
    for i, r in enumerate(top(rows, 'fps_per_w'), 1):
        P(f"| {i} | {r['model']} | {r['category'][:5]} | "
          f"{r['cam_fps']} | {r['power_load_w']} | {r['fps_per_w']} |")
    P("")
    
    if cls:
        P("### Top 10 Classification by Top-1 Accuracy")
        P("")
        P(f"(Sample: up to {N_ACC_IMAGES} ImageNet images; +/- ~3% noise floor)")
        P("")
        P("| Rank | Model | Top-1 | DPU FPS | Cam FPS | Power (W) |")
        P("|---|---|---:|---:|---:|---:|")
        for i, r in enumerate(top(cls, 'accuracy_top1'), 1):
            P(f"| {i} | {r['model']} | {r['accuracy_top1']} | "
              f"{r['dpu_fps']} | {r['cam_fps']} | {r['power_load_w']} |")
        P("")
    
    # ── Detailed tables ──
    def detail_section(title, items, map_lookup):
        if not items: return
        P(f"## Detailed Results -- {title}")
        P("")
        is_det = title.lower().startswith('det')
        head = "| Model | Input | DPU lat mean / p50 / p99 (ms) | DPU FPS | Pwr idle / load (W) |"
        if is_det:
            head += " mAP@0.5 / @0.5:0.95 |"
        else:
            head += " Top-1 |"
        head += " Cam res @ tgt | Cam FPS | cap / pre / inf (ms) | Bound | FPS/W |"
        P(head)
        sep = "|---|---|---|---:|---|"
        sep += "---|" if is_det else "---:|"
        sep += "---|---:|---|:---:|---:|"
        P(sep)
        for r in sorted(items, key=lambda x: -(to_float(x.get('dpu_fps')) or 0)):
            if r['error']:
                P(f"| **{r['model']}** | {r['input_shape']} | ERROR: {r['error'][:40]} |  |  |  |  |  |  |  |  |")
                continue
            lat = f"{r['dpu_latency_mean_ms']} / {r['dpu_latency_p50_ms']} / {r['dpu_latency_p99_ms']}"
            pwr = f"{r['power_idle_w']} / {r['power_load_w']}"
            cres = f"{r.get('cam_resolution') or '-'} @ {r.get('cam_target_fps') or '-'}"
            stages = (f"{r.get('cam_capture_ms') or '-'} / "
                       f"{r.get('cam_preprocess_ms') or '-'} / "
                       f"{r.get('cam_inference_ms') or '-'}")
            bound = ('cam' if r.get('cam_limited') in ('True','true','1')
                     else ('model' if r.get('cam_limited') in ('False','false','0') else '-'))
            if is_det:
                m = map_lookup.get(r['model'], {})
                acc = f"{m.get('map_50') or '-'} / {m.get('map_50_95') or '-'}"
            else:
                acc = r.get('accuracy_top1') or '-'
            P(f"| {r['model']} | {r['input_shape']} | {lat} | {r['dpu_fps']} | {pwr} | "
              f"{acc} | {cres} | {r.get('cam_fps') or '-'} | {stages} | {bound} | "
              f"{r.get('fps_per_w') or '-'} |")
        P("")
    
    detail_section("Classification", cls, {})
    # Detection: pick map from coco first, fall back to voc
    det_map_lookup = {}
    for r in det:
        if r.get('dataset') == 'coco':
            det_map_lookup[r['model']] = coco_by_name.get(r['model'], {})
        elif r.get('dataset') == 'voc':
            det_map_lookup[r['model']] = voc_by_name.get(r['model'], {})
    detail_section("Detection", det, det_map_lookup)
    
    if err:
        P("## Errors")
        P("")
        for r in err:
            P(f"- **{r['model']}** ({r['category']}): `{r['error']}`")
        P("")
    
    REPORT_MD.write_text("\n".join(out), encoding='utf-8')
    print(f"Report written: {REPORT_MD} ({len(out)} lines, {REPORT_MD.stat().st_size:,} bytes)")

generate_combined_report()


## 16. Quick inline view

Prints a compact table of the main benchmark results, sorted by DPU FPS.


In [ ]:
def show_quick():
    if not RESULTS_CSV.exists():
        print('No results yet'); return
    rows = list(csv.DictReader(open(RESULTS_CSV)))
    rows.sort(key=lambda r: to_float(r.get('dpu_fps'), 0), reverse=True)
    hdr = (f"  {'model':<36s} {'cat':<5s} {'input':<14s} "
            f"{'DpuFPS':>7s} {'CamFPS':>7s} {'PwrW':>6s} {'FPS/W':>6s} {'Top1':>6s}")
    print(hdr); print('-'*len(hdr))
    for r in rows:
        if r['error']:
            print(f"  {r['model'][:36]:<36s} ERR {r['error'][:60]}"); continue
        print(f"  {r['model'][:36]:<36s} {r['category'][:4]:<5s} "
              f"{r['input_shape']:<14s} "
              f"{fmt(r.get('dpu_fps')):>7s} {fmt(r.get('cam_fps')):>7s} "
              f"{fmt(r.get('power_load_w')):>6s} {fmt(r.get('fps_per_w')):>6s} "
              f"{fmt(r.get('accuracy_top1')):>6s}")

show_quick()
